In [ ]:
"""
📈 VR Performance Analysis
Designed for: ImmersiveSHAP Technical Validation.
Consolidated Figures: Latency Breakdown, Scalability Frontier, VR Comfort,
Hardware Sensitivity, Phase Stress Comparison, and Correlation Heatmap.
"""

import os
import re
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import shutil
from google.colab import files

# ==========================================
# 1. CONFIGURATION
# ==========================================
USE_GOOGLE_DRIVE = True
DRIVE_PATH = "/content/drive/MyDrive/trabajo_grado_maest_telematica/test_data"
OUTPUT_DIR = "/content/drive/MyDrive/trabajo_grado_maest_telematica/analysis_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

N_MAPPING = {'iris': 150, 'cancer': 569, 'california': 20640}

# ==========================================
# 2. CORE PARSERS
# ==========================================
def clean_numeric(val_str):
    if not val_str: return 0.0
    s = str(val_str).strip()
    # If the string contains both dot and comma, assume comma is decimal (Spanish/European locale)
    # or handle standard cleaning.
    if "," in s:
        if "." in s: return float(s.replace(".", "").replace(",", "."))
        return float(s.replace(",", "."))
    return float(s)

def rejoin_locale_commas(fields, expected_count):
    """Reconstructs numbers that were split into multiple fields by locale commas."""
    fields = [f.strip() for f in fields if f.strip()]
    if len(fields) == 2 * expected_count:
        new_fields = []
        for i in range(0, len(fields), 2):
            new_fields.append(fields[i] + "." + fields[i+1])
        return new_fields
    return fields

def parse_unity_log(content):
    metrics = {'rtt_ms': 0.0, 'parse_ms': 0.0, 'engine_ms': 0.0, 'payload_up': 0.0, 't1_start': 0.0}

    # RECV_NET: Expected 2 values (receiveTime, networkDelay)
    recv_lines = re.findall(r"RECV_NET,(.*)", content)
    if recv_lines:
        raw_fields = recv_lines[-1].split(",")
        fields = rejoin_locale_commas(raw_fields, 2)
        if len(fields) >= 2: metrics['rtt_ms'] = clean_numeric(fields[1])

    # DESERIALIZE_PLOT: Expected 1 value
    des_plot = re.findall(r"DESERIALIZE_PLOT,(.*)", content)
    if des_plot:
        fields = rejoin_locale_commas(des_plot[-1].split(","), 1)
        if fields: metrics['parse_ms'] = clean_numeric(fields[0])

    # SEND_REQ: Expected 1 value (startup offset)
    send_req = re.findall(r"SEND_REQ,(.*)", content)
    if send_req:
        fields = rejoin_locale_commas(send_req[-1].split(","), 1)
        if fields: metrics['t1_start'] = clean_numeric(fields[0])

    # Capture payload size: "Mensaje recibido (1220 bytes)"
    payload_matches = re.findall(r"Mensaje recibido \((\d+) bytes\)", content)
    if payload_matches: metrics['payload_up'] = float(payload_matches[-1])

    def get_time_val(pattern):
        ts_match = re.findall(fr"(\d{{2}}:\d{{2}}:\d{{2}}\.\d{{3}}).*?{pattern}", content)
        if ts_match: return datetime.strptime(ts_match[-1], "%H:%M:%S.%f")
        raw_match = re.findall(fr"{pattern},?.*?([\d,.]+)", content)
        return clean_numeric(raw_match[-1]) if raw_match else None

    t_start = get_time_val(r"\[PlotManager\] Rendering")
    t_end = get_time_val(r"Render completed")

    if t_start and t_end:
        if isinstance(t_start, datetime): metrics['engine_ms'] = (t_end - t_start).total_seconds() * 1000
        else: metrics['engine_ms'] = (t_end - t_start) * 1000
    return metrics

def parse_python_log(content):
    def get_val(tag):
        matches = re.findall(fr"{tag},([\d,.]+)", content)
        return clean_numeric(matches[-1]) if matches else 0.0

    # Return raw values. Scaling is handled in process_experiment relative to client.
    return {
        'train_ms': get_val("TRAINING_MS"), 'shap_ms': get_val("SHAP_MS"),
        'total_server_ms': get_val("TOTAL_LATENCY_MS"), 'payload_bytes': get_val("PAYLOAD_SIZE")
    }

def parse_perf_csv(content):
    try:
        df = pd.read_csv(io.StringIO(content.strip()))
        fps_col = [c for c in df.columns if 'frame_rate' in c.lower() or 'fps' in c.lower()][0]
        v_col = 'avg_vertices_per_frame' if 'avg_vertices_per_frame' in df.columns else None

        # --- Phase Detection Logic ---
        pivot_idx = 0
        if v_col and df[v_col].max() > 0:
            visible_frames = df[df[v_col] > (df[v_col].max() * 0.1)].index
            if len(visible_frames) > 0: pivot_idx = visible_frames[0]

        df_p = df.iloc[:pivot_idx] if pivot_idx > 0 else df
        df_i = df.iloc[pivot_idx:] if pivot_idx > 0 else df

        return {
            'Avg_FPS': df[fps_col].mean(),
            'Low_1_FPS': np.percentile(df[fps_col].dropna(), 1),
            'CFL_ms': df['cfl_max_milliseconds'].mean() if 'cfl_max_milliseconds' in df.columns else 0,
            'Avg_Vertices': df[v_col].mean() if v_col else 0,
            'CPU_Pipeline': df_p['cpu_utilization_percentage'].mean() if 'cpu_utilization_percentage' in df.columns else 0,
            'CPU_Active': df_i['cpu_utilization_percentage'].mean() if 'cpu_utilization_percentage' in df.columns else 0,
            'GPU_Pipeline': df_p['gpu_utilization_percentage'].mean() if 'gpu_utilization_percentage' in df.columns else 0,
            'GPU_Active': df_i['gpu_utilization_percentage'].mean() if 'gpu_utilization_percentage' in df.columns else 0,
            'Power_Watts': df['power_wattage'].mean() / 1000.0 if 'power_wattage' in df.columns else 0,
            'Temp_C': df['battery_temperature_celcius'].mean() if 'battery_temperature_celcius' in df.columns else 0
        }
    except: return None

# ==========================================
# 3. DATA CONSOLIDATION
# ==========================================
def process_experiment(data_sources):
    all_runs = []
    file_registry = {}
    for filename, content in data_sources.items():
        fn_clean = filename.lower().replace(".txt", "").replace(".csv", "").replace("-", "_")
        parts = fn_clean.split("_")
        dtype = 'unity' if "unity" in parts[0] else 'python' if "python" in parts[0] else 'perf' if ("perf" in parts[0] or "rend" in parts[0]) else None
        if not dtype: continue
        run = next((p for p in parts if re.match(r"p\d+", p)), "p1")
        ds = "_".join([p for p in parts if p != parts[0] and p != run])
        key = (ds, run)
        if key not in file_registry: file_registry[key] = {}
        file_registry[key][dtype] = content

    for (ds, run), logs in file_registry.items():
        if 'unity' not in logs or 'python' not in logs: continue
        u, p, o = parse_unity_log(logs['unity']), parse_python_log(logs['python']), parse_perf_csv(logs['perf']) if 'perf' in logs else None
        n_points = int(re.search(r"(\d+)", ds).group(1)) if re.search(r"(\d+)", ds) else N_MAPPING.get(ds.split('_')[0], 0)

        # --- Aggressive Unit Balancing ---
        # If server time (p) is larger than client's total round-trip (u), it's likely microseconds.
        server_total = p['total_server_ms']
        if u['rtt_ms'] > 0 and server_total / u['rtt_ms'] > 50: # High factor protection for long tasks
            scale = 1000.0
            p = {k: (v / scale if k != 'payload_bytes' else v) for k, v in p.items()}
            server_total = p['total_server_ms']

        res = {
            'Dataset': ds, 'Run': run, 'N': n_points,
            'Server_Train': p['train_ms'], 'Server_SHAP': p['shap_ms'],
            'Network_RTT': max(0.0, u['rtt_ms'] - server_total),
            'Unity_Parsing_ms': u['parse_ms'], 'Unity_Engine_ms': u['engine_ms'],
            'Total_Wait_ms': u['rtt_ms'] + u['parse_ms'] + u['engine_ms'],
            'Payload_KB': p['payload_bytes'] / 1024.0,
            'Request_Wait_ms': u['rtt_ms'] # Clean client round-trip
        }
        if o: res.update(o)
        all_runs.append(res)
    return pd.DataFrame(all_runs)



# ==========================================
# 4. SCIENTIFIC VISUALIZATION (Publication-ready)
# ==========================================

import matplotlib.pyplot as plt
import seaborn as sns


def set_scientific_style():
    plt.rcParams.update({
        # Typography (ajustado para impresión real)
        'font.size': 12,
        'axes.titlesize': 13,
        'axes.labelsize': 12,
        'xtick.labelsize': 11,
        'ytick.labelsize': 11,
        'legend.fontsize': 11,
        'legend.title_fontsize': 11,

        # Academic style
        'font.family': 'serif',
        'figure.dpi': 300,
        'savefig.dpi': 600,

        # Line visibility
        'axes.linewidth': 0.9,
        'lines.linewidth': 1.8
    })


def generate_consolidated_plots(df):
    set_scientific_style()

    ds_order = df.groupby('Dataset')['N'].mean().sort_values().index

    # ==========================================
    # FIG 1: Pipeline Efficiency
    # ==========================================
    plt.figure(figsize=(7.5, 4.5))

    cols = ['Server_Train', 'Server_SHAP', 'Network_RTT',
            'Unity_Parsing_ms', 'Unity_Engine_ms']

    plot_df = df.groupby('Dataset')[cols].mean().reindex(ds_order)

    ax = plot_df.plot(
        kind='bar',
        stacked=False,
        cmap='viridis',
        edgecolor='black',
        width=0.8
    )

    plt.yscale('log')
    ax.set_ylabel('Latency (ms, log scale)', fontsize=12)
    ax.tick_params(axis='x', rotation=20, labelsize=11)
    ax.tick_params(axis='y', labelsize=11)

    ax.legend(
        title='Metric',
        loc='upper left',
        frameon=False,
        fontsize=10,
        title_fontsize=10
    )

    plt.tight_layout(pad=1.2)
    plt.savefig(f"{OUTPUT_DIR}/fig1_pipeline_latency.svg",
                bbox_inches='tight', format='svg')

    # ==========================================
    # FIG 2: Scalability Frontier
    # ==========================================
    df_cal = df[df['Dataset'].str.contains('california', case=False)].sort_values('N')

    if not df_cal.empty:
        fig, ax1 = plt.subplots(figsize=(7.2, 4.5))

        sns.lineplot(
            data=df_cal,
            x='N',
            y='Total_Wait_ms',
            ax=ax1,
            color='blue',
            linewidth=2,
            marker='o',
            label='Latency'
        )

        ax1.set_ylabel('End-to-End Latency (ms)', fontsize=12, color='blue')
        ax1.set_xlabel('N (Dataset Size)', fontsize=12)
        ax1.tick_params(labelsize=11)

        ax2 = ax1.twinx()

        sns.lineplot(
            data=df_cal,
            x='N',
            y='Avg_FPS',
            ax=ax2,
            color='red',
            linewidth=2,
            marker='s',
            label='FPS'
        )

        ax2.set_ylabel('Frame Rate (FPS)', fontsize=12, color='red')
        ax2.tick_params(labelsize=11)

        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()

        ax1.legend(
            lines1 + lines2,
            labels1 + labels2,
            loc='upper center',
            bbox_to_anchor=(0.5, -0.18),
            ncol=2,
            frameon=False,
            fontsize=11
        )

        ax2.get_legend().remove()

        plt.tight_layout(pad=1.2)
        plt.savefig(f"{OUTPUT_DIR}/fig2_scalability_frontier.svg",
                    bbox_inches='tight', format='svg')

    # ==========================================
    # FIG 3: VR Comfort
    # ==========================================
    plt.figure(figsize=(7.2, 5))

    comfort_df = df.groupby('Dataset')[['Avg_FPS', 'Low_1_FPS']].mean().reindex(ds_order)

    ax = comfort_df.plot(
        kind='bar',
        color=['#2ecc71', '#e74c3c'],
        edgecolor='black'
    )

    plt.axhline(60, color='blue', linestyle='--', linewidth=1, label='60 FPS Threshold')
    plt.axhline(72, color='black', linestyle=':', linewidth=1, label='72 Hz Target')

    ax.set_ylabel('FPS', fontsize=12)
    ax.set_ylim(0, 95)
    ax.tick_params(axis='x', rotation=20, labelsize=11)
    ax.tick_params(axis='y', labelsize=11)

    for i, (idx, row) in enumerate(comfort_df.iterrows()):
        status = "SAFE" if row['Low_1_FPS'] >= 60 else "UNSTABLE"
        color = "#27ae60" if row['Low_1_FPS'] >= 60 else "#c0392b"

        plt.text(
            i,
            row['Avg_FPS'] + 1,
            status,
            ha='center',
            fontsize=11,
            fontweight='bold',
            color=color
        )

    ax.legend(frameon=False, fontsize=10)

    plt.tight_layout(pad=1.2)
    plt.savefig(f"{OUTPUT_DIR}/fig3_stability_comfort.svg",
                bbox_inches='tight', format='svg')

    # ==========================================
    # FIG 4: Rendering Impact
    # ==========================================
    plt.figure(figsize=(7.0, 4.5))

    sns.regplot(
        data=df,
        x='Avg_Vertices',
        y='GPU_Active',
        scatter_kws={'s': 80, 'edgecolor': 'black', 'alpha': 0.8},
        line_kws={'color': 'red'}
    )

    plt.tight_layout(pad=1.2)
    plt.savefig(f"{OUTPUT_DIR}/fig4_rendering_impact.svg",
                bbox_inches='tight', format='svg')

    # ==========================================
    # FIG 5: Interaction Impact
    # ==========================================
    plt.figure(figsize=(7.0, 4.5))

    sns.regplot(
        data=df,
        x='CFL_ms',
        y='Low_1_FPS',
        scatter_kws={'s': 80, 'edgecolor': 'black', 'color': 'orange', 'alpha': 0.8},
        line_kws={'color': 'blue'}
    )

    plt.tight_layout(pad=1.2)
    plt.savefig(f"{OUTPUT_DIR}/fig5_interaction_impact.svg",
                bbox_inches='tight', format='svg')

    # ==========================================
    # FIG 6: CPU Phase Comparison
    # ==========================================
    plt.figure(figsize=(7.0, 4.5))

    df_melt = df.melt(
        id_vars='Dataset',
        value_vars=['CPU_Pipeline', 'CPU_Active'],
        var_name='Phase',
        value_name='CPU_Utilization'
    )

    ax = sns.barplot(
        data=df_melt,
        x='Dataset',
        y='CPU_Utilization',
        hue='Phase',
        palette='muted',
        edgecolor='black',
        order=ds_order
    )

    ax.set_ylabel('CPU Utilization (%)', fontsize=12)
    ax.tick_params(axis='x', rotation=20, labelsize=11)
    ax.tick_params(axis='y', labelsize=11)

    ax.legend(frameon=False, fontsize=11)

    plt.tight_layout(pad=1.2)
    plt.savefig(f"{OUTPUT_DIR}/fig6_phase_overhead.svg",
                bbox_inches='tight', format='svg')

    plt.show()




# ==========================================
# 5. DIAGNOSTIC SUMMARY
# ==========================================
def export_scientific_summary(df):
    print("\n" + "="*80)
    print("🏆 SCIENTIFIC PERFORMANCE VERDICT")
    print("="*80)

    # Diagnostics
    gpu_corr = df['Avg_Vertices'].corr(df['GPU_Active'])
    n_gpu_corr = df['N'].corr(df['GPU_Active'])
    n_cpu_corr = df['N'].corr(df['CPU_Active'])
    net_sens = df['Payload_KB'].corr(df['Network_RTT']) if 'Payload_KB' in df.columns and df['Payload_KB'].std() > 0 else 0
    interaction_hitch = (df['Avg_FPS'] - df['Low_1_FPS']).mean()

    print(f"📊 Geometry Sensitivity (Vertices vs GPU): {gpu_corr:.2f}")
    print(f"📊 Rendering Sensitivity (N vs GPU):      {n_gpu_corr:.2f}")
    print(f"📊 Logic Sensitivity (N vs CPU):          {n_cpu_corr:.2f}")
    print(f"📊 Payload Sensitivity (Size vs Network):  {net_sens:.2f}")
    print(f"📊 Mean Interaction Hitch (FPS Loss):     {interaction_hitch:.2f} FPS")

    print("\n🚩 TECHNICAL CONCLUSIONS:")
    if gpu_corr > 0.85 or n_gpu_corr > 0.85:
        print("- PRIMARY CAUSE: RENDERING-BOUND. Geometry complexity/Object count saturates the mobile GPU.")
    if n_cpu_corr > 0.70 and n_gpu_corr < 0.70:
        print("- PRIMARY CAUSE: CPU-BOUND. Main thread logic or data processing is the bottleneck.")
    if net_sens > 0.70:
        print("- PRIMARY CAUSE: NETWORK-BOUND. Latency scales linearly with data volume.")
    if interaction_hitch > 15:
        print("- PRIMARY CAUSE: INTERACTION-BOUND. High stutter (Hitch > 15 FPS) during UI/Interaction events.")

    print("\n📄 TABLE: LATENCY & PAYLOAD SUMMARY (MEAN ± STD)")
    cols_lat = ['Total_Wait_ms', 'Network_RTT', 'Server_Train', 'Server_SHAP', 'Unity_Parsing_ms', 'Unity_Engine_ms', 'Payload_KB']
    stats_lat = df.groupby('Dataset')[cols_lat].agg(['mean', 'std']).round(2)
    display(stats_lat)

    print("\n📄 TABLE: SYSTEM STABILITY & RESOURCES (MEAN ± STD)")
    cols_sys = ['Avg_FPS', 'Low_1_FPS', 'CFL_ms', 'GPU_Active', 'Power_Watts']
    stats_sys = df.groupby('Dataset')[cols_sys].agg(['mean', 'std']).round(2)
    display(stats_sys)

    # Exporting formatted tables for Thesis
    stats_lat.to_csv(f"{OUTPUT_DIR}/scientific_latencies.csv")
    stats_sys.to_csv(f"{OUTPUT_DIR}/scientific_stability.csv")

    # Combined Master Table for Publication
    master_cols = ['Total_Wait_ms', 'Unity_Engine_ms', 'Avg_FPS', 'Low_1_FPS', 'CPU_Active', 'GPU_Active', 'Power_Watts', 'Temp_C']
    master_stats = df.groupby('Dataset')[master_cols].agg(['mean', 'std']).round(2)
    master_stats.to_csv(f"{OUTPUT_DIR}/master_performance_stats.csv")

    print(f"\n💾 Results saved to Drive: {OUTPUT_DIR}")

    # --- AUTOMATIC BROWSER DOWNLOAD (ZIP PACKAGE) ---
    print("\n📥 PACKAGING ALL RESULTS (PLOTS + TABLES)...")
    try:
        from google.colab import files
        zip_base = "/content/scientific_results_package"
        shutil.make_archive(zip_base, 'zip', OUTPUT_DIR)
        files.download(f"{zip_base}.zip")
        print("✅ Consolidated ZIP package triggered! Please check your browser.")
    except Exception as e:
        print(f"⚠️ Could not trigger auto-download: {e}")

def main():
    data = {}
    df_runs = pd.DataFrame()

    if USE_GOOGLE_DRIVE:
        from google.colab import drive
        mount_path = '/content/drive'
        try:
            is_mounted = os.path.ismount(mount_path)
            is_empty = not os.path.exists(mount_path) or not os.listdir(mount_path)

            if not is_mounted:
                if is_empty:
                    drive.mount(mount_path, force_remount=False)
                else:
                    print(f"⚠️ Warning: {mount_path} exists but is not empty. Skipping auto-mount.")
                    print("💡 Tip: If you need Drive, run `!rm -rf /content/drive` in a cell and try again.")

            if os.path.exists(DRIVE_PATH):
                for f in os.listdir(DRIVE_PATH):
                    p = os.path.join(DRIVE_PATH, f)
                    if os.path.isfile(p) and not f.startswith('.'):
                        with open(p, 'r', encoding='utf-8', errors='ignore') as file: data[f] = file.read()
                df_runs = process_experiment(data)
        except Exception as e:
            print(f"❌ Drive Error: {e}")

    if df_runs.empty:
        print("\n📂 DATA SOURCE RECOVERY")
        print("No data found in Drive. Please upload your experiment logs (.txt and .csv) manually.")
        uploaded = files.upload()
        if uploaded:
            data = {f: content.decode('utf-8', errors='ignore') if isinstance(content, bytes) else content
                    for f, content in uploaded.items()}
            df_runs = process_experiment(data)

    if not df_runs.empty:
        generate_consolidated_plots(df_runs)
        export_scientific_summary(df_runs)
    else:
        print("❌ No experimental data found. Please ensure files follow the naming convention: <type>_<dataset>_<run>")

if __name__ == "__main__":
    main()
